# marketsim-v2 — run scenarios on Google Colab

Run the crisis-calibrated market simulator in the cloud, with no local setup. This
notebook clones your GitHub repo, installs the package, runs scenario batteries for
Papers 3 / 5 / 6, and saves the result CSVs to your Google Drive (or Supabase).

**How to use:** Runtime -> Run all, or run each cell top to bottom. Cells are grouped:
setup, sanity check, pick a scenario, run a batch or sweep, save results.

_Vercel is not involved and cannot run these batches — this notebook is the supported
way to execute the simulator._


## 1. Setup — clone the repo and install


In [ ]:
# If the repo is PRIVATE, uncomment the token line and paste a GitHub token
# (github.com -> Settings -> Developer settings -> fine-grained token, Contents: read).
REPO = 'https://github.com/zalliata/phd-mkt-sim-agents.git'
# TOKEN = 'ghp_xxx'; REPO = REPO.replace('https://', f'https://{TOKEN}@')

import os, shutil
if os.path.exists('phd-mkt-sim-agents'):
    shutil.rmtree('phd-mkt-sim-agents')
!git clone -q $REPO
print('cloned:', os.path.exists('phd-mkt-sim-agents'))


In [ ]:
# The Python package lives in the marketsim-v2 subfolder of the repo.
# (If you published marketsim-v2 as the repo root instead, set PKG_DIR = 'phd-mkt-sim-agents'.)
PKG_DIR = 'phd-mkt-sim-agents/marketsim-v2'
import os
assert os.path.exists(os.path.join(PKG_DIR, 'pyproject.toml')), \
    f'pyproject.toml not found in {PKG_DIR} — check PKG_DIR matches your repo layout'
%cd $PKG_DIR
!pip install -q -e .
print('installed')


## 2. Sanity check — run the test suite
Confirms the engine, agents, and scenario registry are healthy before you spend compute.


In [ ]:
!python -m pytest -q


## 3. See what you can run


In [ ]:
!marketsim list                 # all scenarios
# !marketsim list --paper P3    # just one paper's battery
# !marketsim show p3-cost-sweep-adversary   # detail for one scenario


## 4. (Optional) Save results to Google Drive
Mounts your Drive so result CSVs persist after the Colab session ends. Skip this cell
to keep results only in the temporary Colab filesystem (you can still download them).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
RESULTS_DIR = '/content/drive/MyDrive/marketsim-results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('results will be saved under', RESULTS_DIR)


## 5. (Optional) Use Supabase instead of local CSVs
Leave this cell unrun to write local CSVs (default). Fill in your project's URL and
service key to write straight to the same Supabase tables the old app used.
In Colab, prefer the key icon (Secrets) over pasting keys into the notebook.


In [ ]:
import os
# os.environ['SUPABASE_URL'] = 'https://xxxx.supabase.co'
# os.environ['SUPABASE_SERVICE_KEY'] = 'your-service-key'
# !pip install -q supabase
print('Supabase configured:' , bool(os.environ.get('SUPABASE_URL')))


## 6. Run a single scenario (quick smoke test)
One iteration, printed summary — good for checking a scenario behaves before a full batch.


In [ ]:
!marketsim run rq2-adversary-enters --seed 0


## 7. Run a Monte Carlo batch
N seeded iterations of one scenario, written to storage. Start small (e.g. 20) to gauge
timing, then scale to the 100 iterations the papers use.


In [ ]:
SCENARIO = 'p3-cost-sweep-adversary'   # change to any id from `marketsim list`
ITERATIONS = 20                        # bump to 100 for the paper runs
!marketsim batch $SCENARIO --iterations $ITERATIONS --storage local


## 8. Run a parameter sweep (Papers 3 / 5 / 6)
Sweeps walk a grid (transaction cost for P3, adversary market share for P5/P6) and run a
full batch at each grid point. These are the heavy jobs — run them here, not on your laptop.


In [ ]:
# Paper 3 — transaction-cost floor sweep (0-50 bps)
!marketsim sweep p3-cost-sweep-adversary --iterations 50

# Paper 5 — detection feasibility (adversary market share 1-50%)
# !marketsim sweep p5-share-sweep-graph --iterations 50

# Paper 6 — algorithmic composition / phase transition
# !marketsim sweep p6-composition-sweep --iterations 25


## 9. Collect the results
Result CSVs land in `results/<run_id>/` (time_steps, scenario_iterations,
agent_snapshots). Copy them to Drive (if mounted) so they survive the session, or zip
and download.


In [ ]:
import glob, shutil, os
runs = sorted(glob.glob('results/*'))
print(f'{len(runs)} run folders:')
for r in runs[:20]: print(' ', r)

# copy to Drive if it was mounted in step 4
if os.path.exists('/content/drive/MyDrive'):
    dest = '/content/drive/MyDrive/marketsim-results'
    os.makedirs(dest, exist_ok=True)
    for r in runs:
        shutil.copytree(r, os.path.join(dest, os.path.basename(r)), dirs_exist_ok=True)
    print('copied to', dest)


In [ ]:
# ...or zip everything and download to your computer
import shutil
shutil.make_archive('marketsim_results', 'zip', 'results')
from google.colab import files
files.download('marketsim_results.zip')


---
### Tips
- **Timing:** run 20 iterations first; multiply to estimate a 100-iteration batch.
- **Reproducibility:** every run is seeded; the same scenario + seed gives identical output.
- **Free-tier limits:** Colab disconnects idle sessions. For the largest P6 sweeps, split
  the share grid across several runs, or use Colab Pro / a cloud VM.
- **Genuine-LLM runs:** add `--llm anthropic` (and set `ANTHROPIC_API_KEY`) to a `run`/`batch`
  command to use a real model instead of the deterministic scripted adversary. Keep batches
  small in that mode — every decision is an API call.
